**The section below is for merging the basic info: CVE, outbreak, phr, enrollment for K12 students at county level**

The base data can be referenced from data folder. We will merge the data based on county name.

Notice: outbreak data other than 2025 is documented in binary way. If needs to do time series analysis, we will change the data to the actual count

In [5]:
library(tidyverse)
library(dplyr)
outbreak <- read.csv("data/raw/base/count.csv")
cve <- read.csv("data/raw/base/cve.csv")
phr_conv <- read.csv("data/raw/brfss/phr_conversion.csv")
df <- cve %>%
  inner_join(outbreak, by = "County",
             suffix = c("_cve", "_out"))
head(df)

Warning message:
"Paket 'ggplot2' wurde unter R Version 4.4.3 erstellt"
Warning message:
"Paket 'forcats' wurde unter R Version 4.4.1 erstellt"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v forcats   1.0.1     v purrr     1.0.2
v ggplot2   4.0.2     v tibble    3.2.1
v lubridate 1.9.3     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


,County,X2016_cve,X2017_cve,X2018_cve,X2019_cve,X2020_cve,X2021_cve,X2022_cve,X2023_cve,X2024_cve,...,X2016_out,X2017_out,X2018_out,X2019_out,X2020_out,X2021_out,X2022_out,X2023_out,X2024_out,X2025_out
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,...,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,Anderson,0.35%,0.50%,0.75%,0.94%,1.00%,1.11%,1.62%,1.64%,2.05%,...,0,0,0,0,0,0,0,0,0,0
2,Andrews,0.75%,0.90%,1.07%,1.50%,0.39%,1.36%,0.45%,1.57%,1.43%,...,0,0,0,0,0,0,0,0,0,3
3,Angelina,0.70%,0.63%,0.67%,0.82%,0.96%,0.86%,1.03%,1.53%,1.98%,...,0,0,0,0,0,0,0,0,0,0
4,Aransas,1.22%,1.39%,1.49%,1.71%,1.57%,1.51%,0.00%,1.91%,1.91%,...,0,0,0,0,0,0,0,0,0,0
5,Archer,0.61%,0.45%,0.81%,1.43%,1.29%,1.56%,1.68%,2.32%,2.80%,...,0,0,0,0,0,0,0,0,0,0
6,Armstrong,0.59%,1.90%,1.44%,2.35%,3.53%,3.53%,3.77%,4.46%,4.95%,...,0,0,0,0,0,0,0,0,0,0


In [6]:
df2025 <- df %>%
  transmute(
    County,
    cve = `X2025_cve`,
    outbreak = `X2025_out`
  )
head(df2025)

,County,cve,outbreak
,<chr>,<chr>,<int>
1,Anderson,2.54%,0
2,Andrews,1.91%,3
3,Angelina,2.50%,0
4,Aransas,2.06%,0
5,Archer,2.70%,0
6,Armstrong,5.24%,0


In [7]:
df2025$cve <- gsub("%", "", df2025$cve)
df2025$cve <- as.numeric(df2025$cve)
head(df2025)
summary(df2025)

,County,cve,outbreak
,<chr>,<dbl>,<int>
1,Anderson,2.54,0
2,Andrews,1.91,3
3,Angelina,2.50,0
4,Aransas,2.06,0
5,Archer,2.70,0
6,Armstrong,5.24,0


    County               cve            outbreak      
 Length:254         Min.   : 0.000   Min.   :  0.000  
 Class :character   1st Qu.: 1.492   1st Qu.:  0.000  
 Mode  :character   Median : 2.490   Median :  0.000  
                    Mean   : 2.817   Mean   :  2.925  
                    3rd Qu.: 3.765   3rd Qu.:  0.000  
                    Max.   :14.540   Max.   :414.000  

In [9]:
enrollment <- read.csv("data/raw/base/enrollment.csv")

enrollment <- enrollment %>%
  mutate(
    County = County %>%
      str_to_lower() %>%
      str_remove(" county") %>%
      str_trim() %>%
      str_to_title()
  )
enrollment <- enrollment[enrollment$County != "Grand Total", ]
View(enrollment)

,County,Enrollment.Sum
,<chr>,<int>
1,Anderson,7808
2,Andrews,4209
3,Angelina,15649
4,Aransas,2913
5,Archer,2110
6,Armstrong,297
7,Atascosa,9046
8,Austin,6290
9,Bailey,1330


In [ ]:
df2025 <- df2025 %>%
  left_join(enrollment, by = "County") %>%
  transmute(
    County,
    cve,
    outbreak,
    enrollment = Enrollment.Sum
  )


In [11]:
head(df2025)

,County,cve,outbreak,enrollment
,<chr>,<dbl>,<int>,<int>
1,Anderson,2.54,0,7808
2,Andrews,1.91,3,4209
3,Angelina,2.50,0,15649
4,Aransas,2.06,0,2913
5,Archer,2.70,0,2110
6,Armstrong,5.24,0,297


In [12]:
df2025 <- df2025 %>%
  inner_join(phr_conv, by = "County") %>%
    transmute(
      County,
      cve,
      outbreak,
      enrollment,
      phr = PHR
    )
head(df2025)
summary(df2025)

,County,cve,outbreak,enrollment,phr
,<chr>,<dbl>,<int>,<int>,<int>
1,Anderson,2.54,0,7808,4
2,Andrews,1.91,3,4209,9
3,Angelina,2.50,0,15649,5
4,Aransas,2.06,0,2913,11
5,Archer,2.70,0,2110,2
6,Armstrong,5.24,0,297,1


    County               cve            outbreak         enrollment    
 Length:254         Min.   : 0.000   Min.   :  0.000   Min.   :    93  
 Class :character   1st Qu.: 1.492   1st Qu.:  0.000   1st Qu.:  1180  
 Mode  :character   Median : 2.490   Median :  0.000   Median :  3240  
                    Mean   : 2.817   Mean   :  2.925   Mean   : 22023  
                    3rd Qu.: 3.765   3rd Qu.:  0.000   3rd Qu.: 10549  
                    Max.   :14.540   Max.   :414.000   Max.   :879002  
                                                       NA's   :5       
      phr        
 Min.   : 1.000  
 1st Qu.: 2.000  
 Median : 5.000  
 Mean   : 5.323  
 3rd Qu.: 8.000  
 Max.   :11.000  
                 

In [ ]:
write.csv(df2025, file = "data/county_data.csv", row.names = FALSE)

**This section is for cleaning the general census data in texas for each county up to need**

In [14]:
# ----- Pull all county-level variables for Texas -----
library(viridis)
library(tidycensus)
census_api_key("d57d54c0ecada0a165fe2386aa8f5078acb68d5b")
texas_counties <- get_acs(
  geography = "county",
  state     = "TX",
  year      = 2024,       
  survey    = "acs5",     
  variables = c(
    # Demographics
    pct_hispanic     = "DP05_0090PE",  # % Hispanic
    pct_black        = "DP05_0045PE",  # % Black non-Hispanic
    pct_white        = "DP05_0037PE",  # % White non-Hispanic
    
    # Socioeconomic
    pct_poverty      = "DP03_0119PE",  # % below poverty line
    pct_uninsured    = "DP03_0099PE",  # % no health insurance
    pct_college      = "DP02_0068PE",  # % college degree+
    median_income    = "DP03_0062PE",   # median household income

    
    # Rural/accessx
    pct_foreign_born = "DP02_0093PE"   # % foreign born
  ),
  output = "wide"   # puts each variable in its own column
)
# Clean it up
texas_counties <- texas_counties %>%
  # Extract county FIPS code
  mutate(county_fips = substr(GEOID, 3, 5)) %>%
  # Keep only the estimate columns (drop margin of error)
  select(GEOID, NAME, county_fips,
         pct_hispanic, pct_black, pct_white,
         pct_poverty, pct_uninsured, 
         pct_college, median_income,
         pct_foreign_born)
head(texas_counties)
summary(texas_counties)

Lade n"otiges Paket: viridisLite

Warning message:
"Paket 'tidycensus' wurde unter R Version 4.4.3 erstellt"
To install your API key for use in future sessions, run this function with `install = TRUE`.

Getting data from the 2020-2024 5-year ACS

Using the ACS Data Profile



GEOID,NAME,county_fips,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,median_income,pct_foreign_born
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
48001,"Anderson County, Texas",001,19.6,18.6,58.3,13.5,18.5,15.4,NA,1.1
48003,"Andrews County, Texas",003,57.5,1.6,57.7,14.2,22.4,17.7,NA,1.2
48005,"Angelina County, Texas",005,23.5,12.1,63.4,11.5,17.7,17.9,NA,1.2
48007,"Aransas County, Texas",007,27.1,1.3,76.9,10.7,12.8,28.8,NA,1.5
48009,"Archer County, Texas",009,9.4,1.5,88.9,4.5,13.9,25.4,NA,0.8
48011,"Armstrong County, Texas",011,12.0,0.6,89.9,6.0,4.2,27.2,NA,0.5


    GEOID               NAME           county_fips         pct_hispanic  
 Length:254         Length:254         Length:254         Min.   : 0.00  
 Class :character   Class :character   Class :character   1st Qu.:19.68  
 Mode  :character   Mode  :character   Mode  :character   Median :27.80  
                                                          Mean   :35.68  
                                                          3rd Qu.:50.02  
                                                          Max.   :97.20  
                                                                         
   pct_black        pct_white      pct_poverty    pct_uninsured  
 Min.   : 0.000   Min.   :14.40   Min.   : 0.00   Min.   : 0.00  
 1st Qu.: 1.200   1st Qu.:55.17   1st Qu.: 7.50   1st Qu.:13.90  
 Median : 3.950   Median :65.80   Median :10.80   Median :16.70  
 Mean   : 6.007   Mean   :63.84   Mean   :11.25   Mean   :16.97  
 3rd Qu.: 8.200   3rd Qu.:75.97   3rd Qu.:13.88   3rd Qu.:19.30  
 Max.   :33.

In [15]:
texas_counties <- texas_counties %>%
  mutate(
    County = NAME %>%
      str_to_lower() %>%
      str_remove(" county, texas") %>%
      str_trim() %>%
      str_to_title()
  )
head(texas_counties)

GEOID,NAME,county_fips,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,median_income,pct_foreign_born,County
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
48001,"Anderson County, Texas",001,19.6,18.6,58.3,13.5,18.5,15.4,NA,1.1,Anderson
48003,"Andrews County, Texas",003,57.5,1.6,57.7,14.2,22.4,17.7,NA,1.2,Andrews
48005,"Angelina County, Texas",005,23.5,12.1,63.4,11.5,17.7,17.9,NA,1.2,Angelina
48007,"Aransas County, Texas",007,27.1,1.3,76.9,10.7,12.8,28.8,NA,1.5,Aransas
48009,"Archer County, Texas",009,9.4,1.5,88.9,4.5,13.9,25.4,NA,0.8,Archer
48011,"Armstrong County, Texas",011,12.0,0.6,89.9,6.0,4.2,27.2,NA,0.5,Armstrong


In [16]:
texas_counties <- texas_counties[, !names(texas_counties) %in% c("GEOID", "NAME", "county_fips")]

texas_counties <- texas_counties %>% relocate(County)
head(texas_counties)
write.csv(texas_counties, file = "data/texas_census.csv", row.names = FALSE)

County,pct_hispanic,pct_black,pct_white,pct_poverty,pct_uninsured,pct_college,median_income,pct_foreign_born
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Anderson,19.6,18.6,58.3,13.5,18.5,15.4,NA,1.1
Andrews,57.5,1.6,57.7,14.2,22.4,17.7,NA,1.2
Angelina,23.5,12.1,63.4,11.5,17.7,17.9,NA,1.2
Aransas,27.1,1.3,76.9,10.7,12.8,28.8,NA,1.5
Archer,9.4,1.5,88.9,4.5,13.9,25.4,NA,0.8
Armstrong,12.0,0.6,89.9,6.0,4.2,27.2,NA,0.5


**This section is for cleaning BRFSS data.**
What we did was from the Total row, and col D10, F10,... for the response and extract as sheetname-response.

In [1]:
import pandas as pd
from openpyxl import load_workbook

# ── CONFIGURE ─────────────────────────────────────────────────────────────────
SRC_DIR = "data/raw/brfss"  # directory containing the 11 PHR Excel files
OUT     = "data/brfss_all_categories.csv"
# ─────────────────────────────────────────────────────────────────────────────

# Excel cols D,F,H,J,L,N,P,R,T,V,X → 0-indexed 3,5,7,9,11,13,15,17,19,21,23
PCT_COLS = [3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23]

records = {}  # key: (sheet_name, response_label) → {phr: pct}

for phr in range(1, 12):
    path = f"{SRC_DIR}/2024_PHR{phr}_BRFSS_Summary_Tables.xlsx"
    wb   = load_workbook(path, read_only=True)

    for sheet_name in wb.sheetnames:
        if sheet_name == "Index":
            continue
        rows = list(wb[sheet_name].iter_rows(values_only=True))
        if len(rows) < 10:
            continue

        label_row = rows[7]   # row 8  in Excel — response category labels
        total_row = rows[9]   # row 10 in Excel — Total row

        for col in PCT_COLS:
            label = label_row[col] if col < len(label_row) else None
            pct   = total_row[col] if col < len(total_row) else None
            if not label or str(label).strip() in ("", "None"):
                continue
            key = (sheet_name, str(label).strip())
            if key not in records:
                records[key] = {}
            records[key][phr] = pct

    print(f"PHR {phr:2d} done")

rows_out = []
for (sheet_name, response), phr_vals in records.items():
    row = {"Sheet": f"{sheet_name} - {response}"}
    for phr in range(1, 12):
        row[str(phr)] = phr_vals.get(phr, None)
    rows_out.append(row)

df = pd.DataFrame(rows_out).sort_values("Sheet").reset_index(drop=True)
df.to_csv(OUT, index=False)
print(f"\nSaved: {OUT}  ({len(df):,} rows x {len(df.columns)} cols)")

PHR  1 done
PHR  2 done
PHR  3 done
PHR  4 done
PHR  5 done
PHR  6 done
PHR  7 done
PHR  8 done
PHR  9 done
PHR 10 done
PHR 11 done

Saved: data/brfss_all_categories.csv  (391 rows x 12 cols)


**This section is for merging all the data as a file**
The file is with base info, full BRFSS data(with total row), and census data. We will join SVI data if needed.
Notice: unfortunately, PHR is called "phr" in county_data.csv if we follow the script, so we need to manually change to "PHR"

In [1]:
library(dplyr)
library(tidyr)
library(readr)
library(stringr)

county_data      <- read_csv("data/county_data.csv")
texas_census     <- read_csv("data/texas_census.csv")
brfss_wide       <- read_csv("data/brfss_all_categories.csv")

normalize_county <- function(x) str_to_title(x)

county_data  <- county_data  |> mutate(County = normalize_county(County))
texas_census <- texas_census |> mutate(County = normalize_county(County))

county_merged <- county_data |>
  left_join(texas_census, by = "County")

cat("county_data rows:     ", nrow(county_data), "\n")
cat("After county join:    ", nrow(county_merged), "\n")
cat("Unmatched counties:   ",
    sum(is.na(county_merged$pct_hispanic)), "\n")

brfss_long <- brfss_wide |>
  rename(variable = Sheet) |>          # first col is the BRFSS variable label
  pivot_longer(
    cols      = -variable,
    names_to  = "PHR",
    values_to = "value"
  ) |>
  mutate(
    PHR   = as.integer(PHR),
    value = suppressWarnings(as.numeric(value))  # "R" and "N" become NA
  )

brfss_county <- brfss_long |>
  pivot_wider(names_from = variable, values_from = value)


final <- county_merged |>
  left_join(brfss_county, by = "PHR")

cat("Final dimensions:     ", nrow(final), "rows x", ncol(final), "cols\n")


write_csv(final, "data/merged_data.csv")


Attache Paket: 'dplyr'


Die folgenden Objekte sind maskiert von 'package:stats':

    filter, lag


Die folgenden Objekte sind maskiert von 'package:base':

    intersect, setdiff, setequal, union


Rows: 254 Columns: 5
-- Column specification --------------------------------------------------------
Delimiter: ","
chr (1): County
dbl (4): cve, outbreak, enrollment, PHR

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 254 Columns: 9
-- Column specification --------------------------------------------------------
Delimiter: ","
chr (1): County
dbl (7): pct_hispanic, pct_black, pct_white, pct_poverty, pct_uninsured, pct...
lgl (1): median_income

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 391 Columns: 12
-- Column specification ----------------------------------

county_data rows:      254 
After county join:     254 
Unmatched counties:    0 
Final dimensions:      254 rows x 404 cols


with SVI data

In [8]:
library(dplyr)
library(readr)

merged  <- read_csv("data/merged_data.csv")
svi     <- read_csv("data/svi_interactive_map.csv")

svi <- svi |>
  mutate(County = gsub(" County", "", COUNTY)) |>
  select(-ST, -STATE, -ST_ABBR, -STCNTY, -COUNTY, -FIPS, -LOCATION, -GeoLevel, -Comparison)

final <- merged |>
  left_join(svi, by = "County")

write_csv(final, "data/merged_with_svi.csv")

Rows: 254 Columns: 404
-- Column specification --------------------------------------------------------
Delimiter: ","
chr   (1): County
dbl (380): cve, outbreak, enrollment, PHR, pct_hispanic, pct_black, pct_whit...
lgl  (23): median_income, Advised to Cut Down Salt - Do not use salt, Diabet...

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 254 Columns: 160
-- Column specification --------------------------------------------------------
Delimiter: ","
chr   (6): STATE, ST_ABBR, COUNTY, LOCATION, GeoLevel, Comparison
dbl (154): ST, STCNTY, FIPS, AREA_SQMI, E_TOTPOP, M_TOTPOP, E_HU, M_HU, E_HH...

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.
